In [1]:
import torch

x = torch.tensor(3.0, requires_grad=True)
loss1 = x ** 2
# 第一次计算梯度，梯度值存储在.grad属性中。
loss1.backward()
print(x.grad)
loss2 = x ** 3
# 第二次计算梯度，注意梯度值不会清零，而是累积到以前的梯度值中。
# retain_graph：是否反向传播后，保留计算图，默认为False。
loss2.backward(retain_graph=True)
print(x.grad)
# 如果我们不希望梯度累积，可以事先对梯度值清零。_ 表示就地修改。
x.grad.zero_()
loss2.backward()
print(x.grad)

tensor(6.)
tensor(33.)
tensor(27.)


In [2]:
def f1():
    x = torch.tensor(1.0, requires_grad=True)
    # 禁用梯度追踪。
    # x.requires_grad_(False)
    x.requires_grad = False
    y = x + 1
    print(y.requires_grad)

def f2():
    x = torch.tensor(1.0, requires_grad=True)
    # 进入无梯度环境，关闭梯度追踪。
    with torch.no_grad():
        y = x + 2
        z = y + 1
        print(y.requires_grad)
        print(z.requires_grad)
    # 退出无梯度环境，恢复梯度追踪。
    w = x * 3
    print(w.requires_grad)


@torch.no_grad()
def f3():
    x = torch.tensor(1.0, requires_grad=True)
    y = x + 1
    print(y.requires_grad)


def f4():
    # 全局禁用梯度追踪。
    torch.set_grad_enabled(False)
    x = torch.tensor(1.0, requires_grad=True)
    y = x + 1
    print(y.requires_grad)
    # 全局启用梯度追踪。
    torch.set_grad_enabled(True)

def f5():
    x = torch.tensor(1.0, requires_grad=True)
    y = x * 2
    # 分离张量，断开梯度追踪。
    z = y.detach()
    print(z.requires_grad)
    # 尝试反向传播，会产生错误。
    # z.backward()
    y.backward()
    print(x.grad)
    
f5()

False
tensor(2.)


In [3]:
from sklearn.datasets import make_regression

x, y, coef = make_regression(n_samples=100, n_features=1, noise=1, random_state=23, coef=True)
x = x.ravel()
print("真实值：", coef)

x = torch.from_numpy(x)
y = torch.from_numpy(y)
w = torch.rand(1, requires_grad=True)
b = torch.rand(1, requires_grad=True)
learning_rate = 0.1

for i in range(30):
    y_pred = x * w + b
    loss = torch.mean((y - y_pred) ** 2)
    print(loss.item())
    loss.backward()
    with torch.no_grad():
        # 对参数的更新，不允许在梯度跟踪的环境中进行。
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        # 对梯度值清零处理。
        w.grad.zero_()
        b.grad.zero_()
print("预测值：")
print(w, b)

真实值： 62.72324997572558
3462.530545739105
2309.3000012072384
1545.0783195341523
1036.9558721583733
698.0188555394374
471.2318131671641
319.03381232784733
216.60383009519296
147.48331963285
100.72281820010274
69.01443019800853
47.46570360898516
32.79152371930741
22.779980506673304
15.937688789716837
11.253972073953628
8.04320450167171
5.839259998631519
4.324597016796727
3.282496242091964
2.564822652959474
2.0701405619334117
1.7288826489914966
1.4932946219100853
1.3305524507948565
1.218062366472449
1.140270809360026
1.0864471000807143
1.0491918050364173
1.0233938791715358
预测值：
tensor([62.5946], requires_grad=True) tensor([0.0287], requires_grad=True)


整理数据集

In [4]:
"""
1. 整理数据集。
将数据转换为PyTorch需要的类型（张量类型），给出数据分批次管理。
2. 编写模型类。
在类中定义神经网络的结构，给出前向传播的逻辑。不需要给出反向传播的实现，因为PyTorch能够自动实现。
3. 训练模型。
在训练模型时，按照以下步骤，循环进行：
    * 获取一个批次的样本数据。
    * 将该批次数据输入给模型，实现前向传播。
    * 定义损失函数，计算误差。
    * 实现反向传播，计算梯度。
    * 根据梯度值，更新参数。
4. 模型的评估。
通常与模型训练同步进行，在训练若干步（step，更新一次参数计数为1个step），进行一次评估。
"""

'\n1. 整理数据集。\n将数据转换为PyTorch需要的类型（张量类型），给出数据分批次管理。\n2. 编写模型类。\n在类中定义神经网络的结构，给出前向传播的逻辑。不需要给出反向传播的实现，因为PyTorch能够自动实现。\n3. 训练模型。\n在训练模型时，按照以下步骤，循环进行：\n    * 获取一个批次的样本数据。\n    * 将该批次数据输入给模型，实现前向传播。\n    * 定义损失函数，计算误差。\n    * 实现反向传播，计算梯度。\n    * 根据梯度值，更新参数。\n4. 模型的评估。\n通常与模型训练同步进行，在训练若干步（step，更新一次参数计数为1个step），进行一次评估。\n'

In [5]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def generate_data(n_samples, n_features, random_state=None):
    """生成随机数据，用于多元线性回归任务。

    Parameters
    ----------
    n_samples : int
        样本数量。
    n_features : int
        特征数量。
    random_state : int
        随机种子。

    Returns
    -------
    train_loader : DataLoader
        训练集的DataLoader对象。
    test_loader : DataLoader
        测试集的DataLoader对象。
    """
    X, y = make_regression(n_samples, n_features, noise=0.1, random_state=random_state)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)
    # 将训练集与测试集包装成DataLoader对象。
    train_loader = get_dataloader(X_train, y_train)
    test_loader = get_dataloader(X_test, y_test, shuffle=False)
    return train_loader, test_loader


def get_dataloader(X, y, batch_size=32, shuffle=True):
    """将数据转换为PyTorch中的DataLoader对象，便于训练与评估模型。

    Parameters
    ----------
    X : array-like, shape=(n_samples, n_features)
        样本数据。
    y : array-like, shape=(n_samples, )
        样本标签。
    batch_size : int
        一个批次的样本数量。
    shuffle : bool
        是否对数据集进行洗牌。
    """
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    # 将多个tensor组合成一个整体，形成一个可以迭代的数据集。
    dataset = TensorDataset(X_tensor, y_tensor)
    # 将Dataset放到Dataloader当中，这样就可以分批次获取数据。
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return dataloader


train_loader, test_loader = generate_data(100, 5, random_state=2)

In [6]:
it = iter(train_loader)
X_batch, y_batch = next(it)
print(X_batch.shape, y_batch.shape)
X_batch, y_batch = next(it)
print(X_batch.shape, y_batch.shape)
X_batch, y_batch = next(it)
print(X_batch.shape, y_batch.shape)
# X_batch, y_batch = next(it)

torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([6, 5]) torch.Size([6])


In [7]:
class LinearRegressionModel(nn.Module):
    """用于实现多元线性回归的模型类。

    在PyTorch中，神经网络模型类需要继承nn.Module类。可以方便管理模型参数，定义
    前向传播逻辑，并与PyTorch其他功能集成。
    """
    def __init__(self, in_features, out_features=1):
        # 首先需要进行父类的初始化。
        super().__init__()
        # 定义一个全连接层（线性层）。线性层执行计算： y = x W^T + b
        # in_features：输入维度。
        # out_features：输出维度。
        self.linear = nn.Linear(in_features=in_features, out_features=out_features)

    def forward(self, x):
        """该方法用于实现神经网络的前向传播逻辑。

        当模型接收到数据时，就会调用该方法。实际上，该方法是在__call__方法中调用的。
        __call__方法接收的参数，会传递给forward，forward方法的返回值会作为__call__
        方法的返回值。

        Parameters
        ----------
        x : torch.Tensor, shape=(batch_size, in_features)
            输入数据。

        Returns
        -------
        torch.Tensor, shape=(batch_size, out_features)
            模型输出结果。
        """
        # print("forward执行")
        return self.linear(x)

In [8]:
model = LinearRegressionModel(3, 1)
x = torch.tensor([[1, 2.0, 3.]])
model(x)

tensor([[1.4897]], grad_fn=<AddmmBackward0>)

In [9]:
# 定义超参数。
n_samples = 1000
n_features = 5
learning_rate = 0.01
epochs = 30
# 生成训练数据。
train_loader, test_loader = generate_data(n_samples, n_features, random_state=3)
# 定义模型类对象。
model = LinearRegressionModel(n_features)
# 定义损失函数。
criterion = nn.MSELoss()
# 定义优化器。用于参数更新，梯度清零操作。
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# 训练与评估模型。
for epoch in range(epochs):
    # 开启训练模型。Dropout与BatchNorm这两个层在训练模型与评估模型下，计算的方式不同。
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        # 1. 前向传播。
        # 根据样本数据，计算输出结果。
        y_pred = model(X_batch)
        # 2. 计算损失。
        # 根据预测值与真实值，计算损失。
        loss = criterion(y_pred.flatten(), y_batch)
        # 3. 反向传播。
        # 将优化器（optimizer）中所有参数的梯度清零。
        optimizer.zero_grad()
        # 反向传播，计算模型参数的梯度值，并将梯度值存储在参数对象的.grad属性中。
        loss.backward()
        # 使用优化器更新模型的参数。
        optimizer.step()
        train_loss += loss.item()
    # 计算训练集的平均损失。
    train_loss /= len(train_loader)

    # 开启评估模型。
    model.eval()
    test_loss = 0.0
    # 在模型评估时，没有必要使用计算图对参数的操作进行跟踪。
    # 因此，这里开启无梯度环境。
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred.flatten(), y_batch)
            test_loss += loss.item()
    test_loss /= len(test_loader)

    # 输出每个epoch模型在训练集与测试集上的损失。
    print(f"Epoch {epoch + 1} / {epochs}, Train Loss: {train_loss:.4f} Test Loss: {test_loss:.4f}")

Epoch 1 / 30, Train Loss: 15049.8393 Test Loss: 7989.8275
Epoch 2 / 30, Train Loss: 6217.2376 Test Loss: 3318.7112
Epoch 3 / 30, Train Loss: 2614.1266 Test Loss: 1397.7630
Epoch 4 / 30, Train Loss: 1111.0079 Test Loss: 597.7060
Epoch 5 / 30, Train Loss: 478.3419 Test Loss: 259.0181
Epoch 6 / 30, Train Loss: 208.4630 Test Loss: 113.5185
Epoch 7 / 30, Train Loss: 91.4820 Test Loss: 50.4932
Epoch 8 / 30, Train Loss: 40.6874 Test Loss: 22.6852
Epoch 9 / 30, Train Loss: 18.3058 Test Loss: 10.2767
Epoch 10 / 30, Train Loss: 8.2733 Test Loss: 4.6980
Epoch 11 / 30, Train Loss: 3.7784 Test Loss: 2.1654
Epoch 12 / 30, Train Loss: 1.7391 Test Loss: 1.0058
Epoch 13 / 30, Train Loss: 0.8111 Test Loss: 0.4707
Epoch 14 / 30, Train Loss: 0.3804 Test Loss: 0.2237
Epoch 15 / 30, Train Loss: 0.1831 Test Loss: 0.1087
Epoch 16 / 30, Train Loss: 0.0912 Test Loss: 0.0551
Epoch 17 / 30, Train Loss: 0.0482 Test Loss: 0.0299
Epoch 18 / 30, Train Loss: 0.0278 Test Loss: 0.0182
Epoch 19 / 30, Train Loss: 0.0182 T

In [10]:
model.eval()
with torch.no_grad():
    # 模拟未知数据。
    X_new = torch.randn(1, n_features)
    # 对未知数据进行预测。
    y_pred = model(X_new)
    print(y_pred)

tensor([[-31.8469]])


In [11]:
# 获取模型所有的参数对象。
for param in model.parameters():
    print(param)
    # 参数的形状。
    print(param.shape)
    # 参数的梯度。
    print(param.grad)
    # 参数的数量。
    print(param.numel())
    # 是否计算梯度。
    print(param.requires_grad)
    print("-" * 20)

for name, param in model.named_parameters():
    print(name)
    print(param)
    print("-" * 20)
# 获取输出层的权重与偏置。
print(model.linear.weight)
print(model.linear.bias)
# 获取参数的底层数据。
print(model.linear.weight.data)


Parameter containing:
tensor([[57.6430, 80.5083, 43.9822, 80.3794, 65.8125]], requires_grad=True)
torch.Size([1, 5])
tensor([[-0.0281, -0.0302, -0.0256,  0.0341, -0.0657]])
5
True
--------------------
Parameter containing:
tensor([-0.0066], requires_grad=True)
torch.Size([1])
tensor([0.0243])
1
True
--------------------
linear.weight
Parameter containing:
tensor([[57.6430, 80.5083, 43.9822, 80.3794, 65.8125]], requires_grad=True)
--------------------
linear.bias
Parameter containing:
tensor([-0.0066], requires_grad=True)
--------------------
Parameter containing:
tensor([[57.6430, 80.5083, 43.9822, 80.3794, 65.8125]], requires_grad=True)
Parameter containing:
tensor([-0.0066], requires_grad=True)
tensor([[57.6430, 80.5083, 43.9822, 80.3794, 65.8125]])


In [12]:
class MultiModel(nn.Module):
    """用于多分类的神经网络类。
    """
    def __init__(self, in_features, out_features=1):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        x = self.linear(x)
        # x = F.log_softmax(x)
        return x


n_samples = 1000
n_features = 5
learning_rate = 0.01
epochs = 30
train_loader, test_loader = generate_data(n_samples, n_features)
model = MultiModel(n_features, 3)
# 定义损失函数，多分类使用交叉熵损失函数。
# NLLLoss与CrossEntropyLoss的区别：
# 前者的输入为对数概率（经过LogSoftmax处理后的结果），后者的输入为logits。后者相当于是LogSoftmax + NLLLoss。
# 官方推荐使用后者。
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# 训练与评估模型。
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    for X_batch, y_batch in train_loader:
        output = model(X_batch)
        loss = criterion(output, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        # 计算正确率。
        y_pred = output.argmax(dim=1)
        train_correct += (y_pred == y_batch).sum().item()
    train_loss /= len(train_loader)
    train_accuracy = train_correct / len(train_loader.dataset)

    model.eval()
    test_loss = 0.0
    test_correct = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            output = model(X_batch)
            loss = criterion(output, y_batch)
            test_loss += loss.item()
            y_pred = output.argmax(dim=1)
            test_correct += (y_pred == y_batch).sum().item()
    
    test_loss /= len(test_loader)
    test_accuracy = test_correct / len(test_loader.dataset)

    print(f"Epoch {epoch + 1} / {epochs} "
        f"Train Loss:{train_loss:.4f} Train Accuracy: {train_accuracy:.4f} "
        f"Test Loss:{test_loss:.4f} Test Accuracy: {test_accuracy:.4f}")

class MultiModel(nn.Module):
    """用于多分类的神经网络类。

    使用多层网络结构（增加隐藏层）。
    """
    def __init__(self, in_features, out_features=1):
        super().__init__()
        n_hidden = 64
        self.hidden_layer = nn.Linear(in_features, n_hidden)
        self.output_layer = nn.Linear(n_hidden, out_features)


    def forward(self, x):
        x = self.hidden_layer(x)
        x = F.relu(x)
        x = self.output_layer(x)
        return x

RuntimeError: expected scalar type Long but found Float

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear=nn.Linear(1,1)
    def forward(self,x):
        return self.linear(x)
model=Model()
criterion=nn.MSELoss()
optimizer=optim.SGD(model.parameters(),lr=0.01)
inputs=torch.tensor([[1.],[2.],[4.],[8.]])
targets=torch.tensor([[2.],[4.],[8.],[16.]])
epochs=100
for epoch in range(epochs):
    outputs=model(inputs)
    loss=criterion(outputs,targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if(epoch+1)%10==0:
        print(f'epoch{epoch+1}/{epoch}loss:{loss.item():.4f}')
        

epoch10/9loss:0.0523
epoch20/19loss:0.0417
epoch30/29loss:0.0365
epoch40/39loss:0.0320
epoch50/49loss:0.0281
epoch60/59loss:0.0246
epoch70/69loss:0.0216
epoch80/79loss:0.0189
epoch90/89loss:0.0166
epoch100/99loss:0.0145


In [ ]:
save_path='model.pth'
dict_=model.state_dict()
print(dict_)
torch.save(dict_,save_path)
print(torch.load(save_path))
load_model=Model()
print('before recover')
print(load_model.linear.weight)
print(load_model.linear.bias)
d=torch.load(save_path)
load_model.load_state_dict(d)
print('after recover')
print(load_model.linear.weight)
print(load_model.linear.bias)
load_model.eval()
with torch.no_grad():
    test_inputs=torch.tensor([5.])
    pred=load_model(test_inputs)
    print('pred:')

OrderedDict([('linear.weight', tensor([[1.9631]])), ('linear.bias', tensor([0.2059]))])
OrderedDict([('linear.weight', tensor([[1.9631]])), ('linear.bias', tensor([0.2059]))])
before recover
Parameter containing:
tensor([[-0.7266]], requires_grad=True)
Parameter containing:
tensor([-0.7294], requires_grad=True)
after recover
Parameter containing:
tensor([[1.9631]], requires_grad=True)
Parameter containing:
tensor([0.2059], requires_grad=True)
